## Phase 1: Data Preparation & Understanding

In [18]:
# Import necessary libraries
import pandas as pd
import numpy as np
import re
import os
import math
import json
from datetime import datetime
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
# import ftfy

# Define file paths
# Get the current directory (where the notebook is located)
current_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in globals() else os.getcwd()

# Construct file paths
data_file = os.path.join(current_dir, 'data', 'realestate_data_london_2024_nov.csv')
output_dir = os.path.join(current_dir, 'output')
output_file = os.path.join(output_dir, 'df_cleaned.csv')

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

### 1.1 Load and Explore the Dataset

In [19]:
# Load the dataset
print("Loading dataset...")
try:
    df = pd.read_csv(data_file, encoding="utf-8")
    print(f" Dataset loaded successfully from: {data_file}")
except FileNotFoundError:
    print(f" Error: File not found at {data_file}")
    print("Current working directory:", os.getcwd())
    print("Available files in data directory:")
    data_dir = os.path.join(current_dir, 'data')
    if os.path.exists(data_dir):
        print(os.listdir(data_dir))
    raise

Loading dataset...
 Dataset loaded successfully from: c:\Users\Admin\Python\S8_Thesis_1\llm\data\realestate_data_london_2024_nov.csv


In [20]:
# Display initial dataset information
print(f"\n1. Dataset shape: {df.shape}")
print(f"   Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print(f"\n2. Columns:")
for i, col in enumerate(df.columns.tolist(), 1):
    print(f"   {i:2d}. {col}")

print(f"\n3. Missing values check:")
missing_values = df.isnull().sum()
if missing_values.sum() > 0:
    print("   Missing values found:")
    for col, missing_count in missing_values[missing_values > 0].items():
        missing_percent = (missing_count / len(df)) * 100
        print(f"   - {col}: {missing_count} missing ({missing_percent:.2f}%)")
else:
    print("   ✓ No missing values found in any column")

print(f"\n4. Data types:")
print(df.dtypes)

print(f"\n5. First 3 rows of original data:")
print(df.head(3))


1. Dataset shape: (1019, 9)
   Rows: 1019, Columns: 9

2. Columns:
    1. addedOn
    2. title
    3. descriptionHtml
    4. propertyType
    5. sizeSqFeetMax
    6. bedrooms
    7. bathrooms
    8. listingUpdateReason
    9. price

3. Missing values check:
   Missing values found:
   - addedOn: 8 missing (0.79%)
   - sizeSqFeetMax: 150 missing (14.72%)
   - bedrooms: 16 missing (1.57%)
   - bathrooms: 35 missing (3.43%)

4. Data types:
addedOn                 object
title                   object
descriptionHtml         object
propertyType            object
sizeSqFeetMax          float64
bedrooms               float64
bathrooms              float64
listingUpdateReason     object
price                   object
dtype: object

5. First 3 rows of original data:
                 addedOn                                              title  \
0             10/10/2024  8 bedroom house for sale in Winnington Road, H...   
1  Reduced on 24/10/2024  7 bedroom house for sale in Brick Street, Mayf

### 1.2 Clean data

In [21]:
# Data Cleaning Functions
def clean_date(date_str):
    """
    Return '2024' for ALL rows, completely ignoring the original value
    """
    return '2024'  # Always return "2024" for all rows

def clean_description(html_text):
    """Clean description - remove HTML tags and extra whitespace"""
    # Handle missing values - return empty string instead of removing row
    if pd.isna(html_text):
        return ""
    
    # Convert to string
    text = str(html_text)
    
    # Remove HTML tags (preserve content between tags)
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # Replace common HTML entities
    html_entities = {
        '&nbsp;': ' ',
        '&amp;': '&',
        '&lt;': '<',
        '&gt;': '>',
        '&quot;': '"',
        '&#39;': "'",
        '&rsquo;': "'",
        '&lsquo;': "'",
        '&rdquo;': '"',
        '&ldquo;': '"'
    }
    
    for entity, replacement in html_entities.items():
        text = text.replace(entity, replacement)
    
    # Clean up extra whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # Strip leading/trailing whitespace
    return text.strip()

def clean_price(price_value):
    """Clean price - remove currency symbols and commas, convert to float"""
    # Handle missing values - return NaN but don't remove row
    if pd.isna(price_value):
        return np.nan
    
    # Convert to string
    price_str = str(price_value)
    
    # Extract all numbers (including decimals)
    # This preserves the numeric value regardless of format
    numbers = re.findall(r'[\d,\.]+', price_str)
    
    if not numbers:
        return np.nan
    
    # Take the first number found (should be the price)
    price_num = numbers[0]
    
    # Clean the number
    # Remove commas (thousands separators)
    price_num = price_num.replace(',', '')
    
    # Handle cases where dot might be decimal separator
    # If there's a dot and it's not the last character, assume it's decimal
    if '.' in price_num and price_num.rfind('.') < len(price_num) - 1:
        # Already has decimal point, keep as is
        pass
    else:
        # No valid decimal point found
        pass
    
    # Convert to float
    try:
        return float(price_num)
    except:
        # If conversion fails, try to handle special cases
        try:
            # Remove any remaining non-numeric characters
            clean_num = re.sub(r'[^\d\.]', '', price_str)
            return float(clean_num) if clean_num else np.nan
        except:
            return np.nan      

In [22]:
# Apply data cleaning
# Create a copy for cleaning
df_cleaned = df.copy()

# 1 Clean and rename 'addedOn' column
print("\n1. Cleaning 'addedOn' column...")
df_cleaned['Date'] = df_cleaned['addedOn'].apply(clean_date)
df_cleaned = df_cleaned.drop('addedOn', axis=1)

# 2 Clean and rename 'descriptionHtml' column
print("\n2. Cleaning 'descriptionHtml' column...")
df_cleaned['listingDescription'] = df_cleaned['descriptionHtml'].apply(clean_description)
df_cleaned = df_cleaned.drop('descriptionHtml', axis=1)

# Calculate some statistics about the cleaned descriptions
desc_lengths = df_cleaned['listingDescription'].apply(len)
print(f"    HTML tags removed")
print(f"   Average description length: {desc_lengths.mean():.0f} characters")
print(f"   Min length: {desc_lengths.min()} characters")
print(f"   Max length: {desc_lengths.max()} characters")

# 3 Clean 'price' column
print("\n3. Cleaning 'price' column...")
original_price_sample = df_cleaned['price'].head(3).tolist()
df_cleaned['price'] = df_cleaned['price'].apply(clean_price)

# Remove rows with invalid prices
original_rows = len(df_cleaned)
df_cleaned = df_cleaned.dropna(subset=['price'])
rows_removed = original_rows - len(df_cleaned)

print(f"   Currency symbols and commas removed")
print(f"   Rows with invalid prices removed: {rows_removed}")
print(f"   Price range: £{df_cleaned['price'].min():,.2f} to £{df_cleaned['price'].max():,.2f}")
print(f"   Average price: £{df_cleaned['price'].mean():,.2f}")

# 4 Remove rows with missing values in key numeric features
key_cols = ['sizeSqFeetMax', 'bedrooms', 'bathrooms']
rows_before_dropna = len(df_cleaned)
df_cleaned = df_cleaned.dropna(subset=key_cols)
rows_removed_nulls = rows_before_dropna - len(df_cleaned)
print(f"\n4. Rows with missing values in {key_cols} removed: {rows_removed_nulls}")

# 5 Check other columns
print("\n5. Checking other columns...")

# Check for missing values in other columns
missing_after_clean = df_cleaned.isnull().sum()
if missing_after_clean.sum() > 0:
    print("   Missing values after cleaning:")
    for col, missing_count in missing_after_clean[missing_after_clean > 0].items():
        print(f"   - {col}: {missing_count} missing")
else:
    print("    No missing values in cleaned data")


1. Cleaning 'addedOn' column...

2. Cleaning 'descriptionHtml' column...
    HTML tags removed
   Average description length: 1616 characters
   Min length: 106 characters
   Max length: 9210 characters

3. Cleaning 'price' column...
   Currency symbols and commas removed
   Rows with invalid prices removed: 1
   Price range: £315,000.00 to £80,000,000.00
   Average price: £11,298,546.61

4. Rows with missing values in ['sizeSqFeetMax', 'bedrooms', 'bathrooms'] removed: 168

5. Checking other columns...
    No missing values in cleaned data


In [23]:
# Display cleaned dataset information
print(f"\n1. Cleaned dataset shape: {df_cleaned.shape}")
print(f"   Rows: {df_cleaned.shape[0]}, Columns: {df_cleaned.shape[1]}")

print(f"\n2. Cleaned columns:")
for i, col in enumerate(df_cleaned.columns.tolist(), 1):
    print(f"   {i:2d}. {col} (dtype: {df_cleaned[col].dtype})")

print(f"\n3. Sample of cleaned data (first 2 rows):")
print(df_cleaned.head(2))

print(f"\n4. Data types summary:")
print(df_cleaned.dtypes)

print(f"\n5. Basic statistics for numeric columns:")
numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    print(df_cleaned[numeric_cols].describe())
else:
    print("   No numeric columns found")


1. Cleaned dataset shape: (850, 9)
   Rows: 850, Columns: 9

2. Cleaned columns:
    1. title (dtype: object)
    2. propertyType (dtype: object)
    3. sizeSqFeetMax (dtype: float64)
    4. bedrooms (dtype: float64)
    5. bathrooms (dtype: float64)
    6. listingUpdateReason (dtype: object)
    7. price (dtype: float64)
    8. Date (dtype: object)
    9. listingDescription (dtype: object)

3. Sample of cleaned data (first 2 rows):
                                               title propertyType  \
0  8 bedroom house for sale in Winnington Road, H...        House   
1  7 bedroom house for sale in Brick Street, Mayf...        House   

   sizeSqFeetMax  bedrooms  bathrooms listingUpdateReason       price  Date  \
0        16749.0       8.0        8.0                 new  24950000.0  2024   
1        12960.0       7.0        7.0       price_reduced  29500000.0  2024   

                                  listingDescription  
0  This magnificent home, set behind security gat...  
1  In 

### 1.3 Save cleaned data

In [24]:
# Save cleaned data
df_cleaned.to_csv(output_file, index=False)
print(f" Cleaned data saved to: {output_file}") 

 Cleaned data saved to: c:\Users\Admin\Python\S8_Thesis_1\llm\output\df_cleaned.csv


## Phase 2: Use Zero-shot to extract feature scores

In [25]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Zero-Shot Entailment Feature Scoring for Real Estate Listings
Fixed version with better error handling and compatibility
"""

import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Force CPU mode to avoid GPU compatibility issues
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# Now import torch
import torch
from transformers import pipeline
from tqdm import tqdm
import time

# ============================================
# CONFIGURATION
# ============================================
print("=" * 70)
print("ZERO-SHOT ENTAILMENT FEATURE SCORING FOR REAL ESTATE LISTINGS")
print("=" * 70)

# Set paths - FIXED THE PATH
current_dir = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
# Fix the path - remove extra "llm" duplication
output_dir = os.path.join(current_dir, "output")
input_file = os.path.join(output_dir, "df_cleaned.csv")
output_file = os.path.join(output_dir, "df_feature_extract_zero_shot_score.csv")

print(f"Current directory: {current_dir}")
print(f"Output directory: {output_dir}")
print(f"Input file: {input_file}")

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# ============================================
# LOAD INPUT DATA
# ============================================
print(f"\n[1/5] Loading input data...")

try:
    df = pd.read_csv(input_file, encoding='utf-8')
    print(f"✓ Loaded {len(df)} rows and {len(df.columns)} columns")
    print(f"✓ Columns: {', '.join(df.columns.tolist())}")
except FileNotFoundError:
    print(f"✗ Error: Input file not found at {input_file}")
    print("Please ensure df_cleaned.csv exists in the output directory.")
    exit(1)
except Exception as e:
    print(f"✗ Error loading file: {e}")
    exit(1)

# ============================================
# INITIALIZE ZERO-SHOT CLASSIFIER
# ============================================
print(f"\n[2/5] Loading zero-shot entailment model...")

# Use a more compatible model
model_name = "cross-encoder/nli-distilroberta-base"  # More compatible than BART
print(f"✓ Loading model: {model_name}")
print("  (This may take 30-60 seconds on first run)")

try:
    classifier = pipeline(
        "zero-shot-classification",
        model=model_name,
        device=-1,  # Force CPU
    )
    print("✓ Model loaded successfully!")
except Exception as e:
    print(f"✗ Error loading model: {e}")
    print("Trying fallback to another model...")
    try:
        # Last resort - tiny model
        classifier = pipeline(
            "zero-shot-classification",
            model="typeform/distilbert-base-uncased-mnli",
            device=-1,
        )
        print("✓ Fallback model loaded successfully!")
    except Exception as e:
        print(f"✗ Critical error loading model: {e}")
        exit(1)

# ============================================
# DEFINE FEATURE HYPOTHESES
# ============================================
print(f"\n[3/5] Defining feature hypotheses...")

feature_hypotheses = {
    'luxury': [
        "This property has luxury amenities.",
        "This property offers high-end features.",
        "This is a luxury home.",
        "This property has premium finishes.",
        "This property includes concierge or pool facilities.",
    ],
    
    'transport': [
        "This property has good transport links.",
        "This property is close to public transportation.",
        "This property is near a tube station.",
        "This property has excellent connectivity.",
        "This property is within walking distance of transport.",
    ],
    
    'school': [
        "This property is near good schools.",
        "This property is in a good school catchment area.",
        "This property has access to excellent schools.",
        "This property is close to educational facilities.",
        "This property is well-served by local schools.",
    ],
    
    'renovation': [
        "This property has been recently renovated.",
        "This property is newly refurbished.",
        "This property has been modernized.",
        "This property is in excellent condition.",
        "This property has updated interiors.",
    ]
}

# Combine ALL hypotheses into one list
all_hypotheses = []
for hyps in feature_hypotheses.values():
    all_hypotheses.extend(hyps)

# Track which indices belong to which feature
feature_indices = {}
start_idx = 0
for feature, hyps in feature_hypotheses.items():
    end_idx = start_idx + len(hyps)
    feature_indices[feature] = list(range(start_idx, end_idx))
    start_idx = end_idx

print(f"✓ Combined {len(all_hypotheses)} hypotheses across 4 features")
for feature, hyps in feature_hypotheses.items():
    print(f"  - {feature}: {len(hyps)} hypotheses")

# ============================================
# SCORING FUNCTION
# ============================================
def clean_description(desc):
    """Clean and truncate description for model input."""
    if pd.isna(desc) or not isinstance(desc, str) or desc.strip() == "":
        return None
    
    # Remove common encoding artifacts
    desc = desc.encode('utf-8', errors='ignore').decode('utf-8')
    
    # Truncate if too long
    if len(desc) > 2000:  # Conservative limit
        desc = desc[:2000] + "..."
    
    return desc.strip()

# ============================================
# PROCESS ALL LISTINGS
# ============================================
print(f"\n[4/5] Processing {len(df)} listings with zero-shot entailment...")
print(f"Using ONE model call per listing")

# Initialize score columns
for feature in feature_hypotheses.keys():
    df[f"{feature}_score"] = 0.0

# Process each listing with progress bar
start_time = time.time()
success_count = 0
error_count = 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Scoring listings"):
    description = clean_description(row['listingDescription'])
    
    if description is None:
        error_count += 1
        continue
    
    try:
        # SINGLE model call for ALL hypotheses
        result = classifier(
            description,
            all_hypotheses,
            hypothesis_template="{}",
            multi_label=True
        )
        
        # Extract scores for each feature
        scores = result['scores']
        for feature, indices in feature_indices.items():
            feature_scores = [scores[i] for i in indices]
            df.at[idx, f"{feature}_score"] = round(max(feature_scores), 4)
        
        success_count += 1
        
    except Exception as e:
        error_count += 1
        if error_count <= 5:  # Only show first 5 errors
            print(f"\nWarning: Error processing row {idx}: {type(e).__name__}: {e}")
        continue

elapsed_time = time.time() - start_time
print(f"\n✓ Processing complete in {elapsed_time:.1f} seconds")
print(f"✓ Successful: {success_count} rows")
print(f"✓ Failed: {error_count} rows")

# ============================================
# VERIFY AND SAVE RESULTS
# ============================================
print(f"\n[5/5] Saving results...")

# Display sample of non-zero scores if they exist
print(f"\nSample of scored listings (first 10 rows):")
sample_cols = ['price', 'luxury_score', 'transport_score', 'school_score', 'renovation_score']
sample_df = df[sample_cols].head(10)
print(sample_df.to_string())

# Show statistics
print(f"\nScore statistics:")
for feature in feature_hypotheses.keys():
    col = f"{feature}_score"
    non_zero = (df[col] > 0).sum()
    print(f"  {col}: mean={df[col].mean():.4f}, non-zero={non_zero}/{len(df)}")

# Save to CSV
try:
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"✓ File saved successfully to: {output_file}")
except Exception as e:
    print(f"✗ Error saving file: {e}")

# ============================================
# SUMMARY
# ============================================
print("\n" + "=" * 70)
print("PROCESSING SUMMARY")
print("=" * 70)
print(f"Input file:  {input_file}")
print(f"Output file: {output_file}")
print(f"Listings processed: {len(df)}")
print(f"Successful: {success_count}")
print(f"Failed: {error_count}")
print(f"Processing time: {elapsed_time:.1f} seconds")
print(f"Model used: {model_name}")
print("=" * 70)
print("\n✓ Script completed successfully!")

ZERO-SHOT ENTAILMENT FEATURE SCORING FOR REAL ESTATE LISTINGS
Current directory: c:\Users\Admin\Python\S8_Thesis_1\llm
Output directory: c:\Users\Admin\Python\S8_Thesis_1\llm\output
Input file: c:\Users\Admin\Python\S8_Thesis_1\llm\output\df_cleaned.csv

[1/5] Loading input data...
✓ Loaded 850 rows and 9 columns
✓ Columns: title, propertyType, sizeSqFeetMax, bedrooms, bathrooms, listingUpdateReason, price, Date, listingDescription

[2/5] Loading zero-shot entailment model...
✓ Loading model: cross-encoder/nli-distilroberta-base
  (This may take 30-60 seconds on first run)


The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.


✗ Error loading model: data did not match any variant of untagged enum ModelWrapper at line 250356 column 3
Trying fallback to another model...


The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.


✓ Fallback model loaded successfully!

[3/5] Defining feature hypotheses...
✓ Combined 20 hypotheses across 4 features
  - luxury: 5 hypotheses
  - transport: 5 hypotheses
  - school: 5 hypotheses
  - renovation: 5 hypotheses

[4/5] Processing 850 listings with zero-shot entailment...
Using ONE model call per listing


Scoring listings: 100%|██████████| 850/850 [1:05:07<00:00,  4.60s/it]


✓ Processing complete in 3907.1 seconds
✓ Successful: 850 rows
✓ Failed: 0 rows

[5/5] Saving results...

Sample of scored listings (first 10 rows):
        price  luxury_score  transport_score  school_score  renovation_score
0  24950000.0        0.9439           0.8865        0.8622            0.6884
1  29500000.0        0.9992           0.9971        0.9928            0.9674
2  25000000.0        0.9994           0.9867        0.9651            0.9373
3  24950000.0        0.9565           0.9324        0.9049            0.8654
4  24950000.0        1.0000           0.9932        0.8666            0.3747
5  25000000.0        0.9996           0.9979        0.9932            0.9761
6  25000000.0        0.9998           0.9993        0.9990            0.9972
7  25000000.0        0.9999           0.9998        0.9893            0.8832
8  25000000.0        0.9990           0.9967        0.9943            0.9849
9  25000000.0        0.9968           0.9354        0.6987            0.4400

Sc

## Phase 3: Train & Evaluate Random Forest Models

 - Baseline: df_cleaned.csv (no extracted scores)
 - Enhanced: df_feature_extract_zero_shot_score.csv (with extracted scores)

In [26]:
# Phase 4: Train & Evaluate Random Forest Models

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib

# Function: evaluate model
def evaluate_model(model, X_train, X_test, y_train, y_test, label="model"):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    print(f"\n=== {label} ===")
    print(f"RMSE: {rmse:,.2f}")
    print(f"MAE : {mae:,.2f}")
    return rmse, mae

# 4.1 Baseline model on df_cleaned.csv
current_dir = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
output_dir = os.path.join(current_dir, "output")
cleaned_file = os.path.join(output_dir, "df_cleaned.csv")
df_cleaned = pd.read_csv(cleaned_file)
print("Loaded cleaned data:", cleaned_file)
print("Cleaned columns:", df_cleaned.columns.tolist())

target_col = "price"

# numeric features
num_cols = ["sizeSqFeetMax", "bedrooms", "bathrooms"]
# categorical to encode
cat_cols = ["propertyType", "listingUpdateReason"]

# one‑hot encode categoricals
df_cleaned_cat = pd.get_dummies(df_cleaned[cat_cols], prefix=cat_cols, drop_first=False)
X_base = pd.concat([df_cleaned[num_cols].reset_index(drop=True),
                    df_cleaned_cat.reset_index(drop=True)], axis=1)
y_base = df_cleaned[target_col].copy()

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42
)

rf_base = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rmse_base, mae_base = evaluate_model(
    rf_base, Xb_train, Xb_test, yb_train, yb_test,
    label="Random Forest (baseline: num + encoded propertyType/listingUpdateReason)"
)

baseline_model_path = os.path.join(output_dir, "rf_baseline_model.pkl")
joblib.dump(rf_base, baseline_model_path)
print("Baseline model saved to:", baseline_model_path)

# 4.2 Enhanced model on df_feature_extract_zero_shot_score.csv
#    (must already contain luxury_score, transport_score, school_score, renovation_score)
scores_file = os.path.join(output_dir, "df_feature_extract_zero_shot_score.csv")
df_scores = pd.read_csv(scores_file)
print("\nLoaded data with extracted scores:", scores_file)
print("Columns:", df_scores.columns.tolist())

# numeric + scores
score_cols = ["luxury_score", "transport_score", "school_score", "renovation_score"]
num_cols_scores = ["sizeSqFeetMax", "bedrooms", "bathrooms"] + score_cols

# one‑hot encode same categoricals
df_scores_cat = pd.get_dummies(df_scores[cat_cols], prefix=cat_cols, drop_first=False)

X_scores = pd.concat([df_scores[num_cols_scores].reset_index(drop=True),
                      df_scores_cat.reset_index(drop=True)], axis=1)
y_scores = df_scores[target_col].copy()

Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_scores, y_scores, test_size=0.2, random_state=42
)

rf_scores = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rmse_scores, mae_scores = evaluate_model(
    rf_scores, Xs_train, Xs_test, ys_train, ys_test,
    label="Random Forest (num + encoded categoricals + extracted scores)"
)

scores_model_path = os.path.join(output_dir, "rf_with_scores_zero_shot_model.pkl")
joblib.dump(rf_scores, scores_model_path)
print("Model with scores saved to:", scores_model_path)

# 4.3 Compare performance
print("\n=== Performance comparison (test set) ===")
print(f"Baseline RF - RMSE: {rmse_base:,.2f}, MAE: {mae_base:,.2f}")
print(f"With scores RF - RMSE: {rmse_scores:,.2f}, MAE: {mae_scores:,.2f}")

if rmse_scores < rmse_base:
    print("→ RMSE improved after adding extracted feature scores.")
else:
    print("→ RMSE did not improve after adding extracted feature scores.")

if mae_scores < mae_base:
    print("→ MAE improved after adding extracted feature scores.")
else:
    print("→ MAE did not improve after adding extracted feature scores.")


Loaded cleaned data: c:\Users\Admin\Python\S8_Thesis_1\llm\output\df_cleaned.csv
Cleaned columns: ['title', 'propertyType', 'sizeSqFeetMax', 'bedrooms', 'bathrooms', 'listingUpdateReason', 'price', 'Date', 'listingDescription']

=== Random Forest (baseline: num + encoded propertyType/listingUpdateReason) ===
RMSE: 5,428,669.93
MAE : 3,214,482.12
Baseline model saved to: c:\Users\Admin\Python\S8_Thesis_1\llm\output\rf_baseline_model.pkl

Loaded data with extracted scores: c:\Users\Admin\Python\S8_Thesis_1\llm\output\df_feature_extract_zero_shot_score.csv
Columns: ['title', 'propertyType', 'sizeSqFeetMax', 'bedrooms', 'bathrooms', 'listingUpdateReason', 'price', 'Date', 'listingDescription', 'luxury_score', 'transport_score', 'school_score', 'renovation_score']

=== Random Forest (num + encoded categoricals + extracted scores) ===
RMSE: 5,995,209.01
MAE : 3,687,268.37
Model with scores saved to: c:\Users\Admin\Python\S8_Thesis_1\llm\output\rf_with_scores_zero_shot_model.pkl

=== Performa